In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<Accordion>
<AccordionItem title="Versi pakej">

Kod di halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Kamu boleh menggunakan pilihan untuk menyesuaikan primitif Estimator. Walaupun antara muka kaedah `run()` primitif adalah sama merentasi semua pelaksanaan, pilihan mereka tidak sama. Rujuk rujukan API untuk maklumat tentang pilihan [`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) dan [`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html).

<span id="pass-options"></span>

## Tetapkan pilihan Estimator

Kamu boleh menetapkan pilihan semasa menginisialisasi Estimator, selepas menginisialisasi Estimator, atau kamu boleh mengemas kini pilihan selepas Estimator diinisialisasi. Untuk arahan menggunakan teknik-teknik ini, lihat topik [Pengenalan kepada pilihan](/guides/runtime-options-overview#options-precedence).

Selain itu, kamu boleh menetapkan nilai `precision` dalam kaedah `run()`, seperti yang diterangkan dalam bahagian berikut.
<span id="run-method"></span>

### Kaedah Run()

Satu-satunya nilai yang boleh kamu hantar kepada `run()` adalah yang ditakrifkan dalam antara muka. Iaitu, `precision` untuk Estimator. Ini menimpa sebarang nilai yang ditetapkan untuk `default_precision` untuk run semasa.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Kes khas: ketepatan
Kaedah `EstimatorV2.run` menerima dua argumen: senarai PUB, yang masing-masing boleh menentukan nilai ketepatan khusus PUB, dan argumen kata kunci ketepatan. Nilai ketepatan ini adalah sebahagian daripada antara muka pelaksanaan Estimator, dan bebas daripada pilihan Runtime Estimator. Ia mengambil keutamaan daripada sebarang nilai yang dinyatakan sebagai pilihan untuk mematuhi abstraksi Estimator.

Walau bagaimanapun, jika `precision` tidak dinyatakan oleh mana-mana PUB atau dalam argumen kata kunci run (atau jika semuanya `None`), maka nilai ketepatan daripada pilihan digunakan, terutamanya `default_precision`.

Perlu diambil perhatian bahawa pilihan Estimator mengandungi kedua-dua `default_shots` dan `default_precision`. Walau bagaimanapun, kerana gate-twirling diaktifkan secara lalai, hasil darab `num_randomizations` dan `shots_per_randomization` mengambil keutamaan daripada kedua-dua pilihan tersebut.

Khususnya, untuk mana-mana Estimator PUB:

1. Jika PUB menentukan ketepatan, gunakan nilai tersebut.
2. Jika argumen kata kunci ketepatan dinyatakan dalam `run`, gunakan nilai tersebut.
3. Jika `twirling` diaktifkan (True secara lalai), maka hasil darab `num_randomizations` dan `shots_per_randomization`, seperti yang dinyatakan dalam [pilihan `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options), digunakan.
4. Jika `estimator.options.default_shots` dinyatakan, gunakan nilai tersebut untuk mengawal jumlah data.
5. Jika `estimator.options.default_precision` dinyatakan, gunakan nilai tersebut.

Sebagai contoh, jika ketepatan dinyatakan di keempat-empat tempat, yang mempunyai keutamaan tertinggi (ketepatan yang dinyatakan dalam PUB) digunakan.

> **Note:** Ketepatan berskala songsang dengan penggunaan. Iaitu, semakin rendah ketepatan, semakin banyak masa QPU yang diperlukan untuk dijalankan.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## Matikan semua mitigasi ralat dan penindasan ralat
Kamu boleh mematikan semua mitigasi dan penindasan ralat jika kamu sedang, sebagai contoh, membuat kajian tentang teknik mitigasi sendiri. Untuk melakukan ini, tetapkan `resilience_level = 0`.

Contoh:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## Pilihan yang tersedia
Jadual berikut mendokumentasikan pilihan daripada versi terkini `qiskit-ibm-runtime`. Untuk melihat versi pilihan yang lebih lama, lawati [rujukan API `qiskit-ibm-runtime`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime) dan pilih versi sebelumnya.

<Accordion>
<AccordionItem title="`default_shots`">

Jumlah keseluruhan shots yang digunakan bagi setiap litar bagi setiap konfigurasi.

**Pilihan**: Integer >= 0

**Lalai**: None

[Dokumentasi API `default_shots`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

Ketepatan lalai yang digunakan untuk mana-mana PUB atau panggilan `run()` yang tidak menentukannya.

**Pilihan**: Float > 0

**Lalai**: 0.015625 (1 / sqrt(4096))

[Dokumentasi API `default_precision`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

Kawal tetapan mitigasi ralat dynamical decoupling.

[Dokumentasi API `dynamical_decoupling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Pilihan**: `True`, `False`

**Lalai**: `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Pilihan**: `middle`, `edges`

**Lalai**: `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Pilihan: `asap`, `alap`
Lalai: `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

Pilihan: `XX`, `XpXm`, `XY4`
Lalai: `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Pilihan: `True`, `False`
Lalai: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[Dokumentasi API `environment`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Fungsi boleh panggil yang menerima `Job ID` dan `Job result`.

**Pilihan**: None

**Lalai**: None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

Senarai tag.

**Pilihan**: None

**Lalai**: None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**Pilihan**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Lalai**: WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**Pilihan**: `True`, `False`

**Lalai**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[Dokumentasi API `execution`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Sama ada untuk menetapkan semula qubit ke keadaan asas untuk setiap shot.

**Pilihan**: `True`, `False`

**Lalai**: `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
Kelewatan antara pengukuran dan litar kuantum berikutnya.

**Pilihan**: Nilai dalam julat yang disediakan oleh `backend.rep_delay_range`

**Lalai**: Diberikan oleh `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
Mengehadkan tempoh masa sesuatu kerja boleh berjalan, dalam saat. Lihat panduan [masa pelaksanaan maksimum](/guides/max-execution-time) untuk butiran.

**Pilihan**: Nombor integer saat dalam julat [1, 10800]

**Lalai**: 10800 (3 jam)
  </AccordionItem>

<AccordionItem title="`resilience`">
Pilihan ketahanan lanjutan untuk menala strategi ketahanan dengan tepat.

[Dokumentasi API `resilience`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Pilihan untuk pembelajaran hingar lapisan.

[Dokumentasi API `resilience.layer_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Pilihan**: list[int] daripada 2-10 nilai dalam julat [0, 200]

**Lalai**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Pilihan**: None, Integer >= 1

**Lalai**: `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Pilihan**: Integer >= 1

**Lalai**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Pilihan**: Integer >= 1

**Lalai**: `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**Pilihan**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Lalai**: None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**Pilihan**: `True`, `False`

**Lalai**: `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

Pilihan untuk pembelajaran hingar pengukuran.

[Dokumentasi API `resilience.measure_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Pilihan**: Integer >= 1

**Lalai**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Pilihan**: Integer, `auto`

**Lalai**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**Pilihan**: `True`, `False`

**Lalai**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

Pilihan mitigasi pembatalan ralat berkemungkinan.

[Dokumentasi API `resilience.pec`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**Pilihan**: `None`, Integer >= 1

**Lalai**: `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**Pilihan**: `auto`, float dalam julat [0, 1]

**Lalai**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**Pilihan**: `True`, `False`

**Lalai**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[Dokumentasi API `resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**Pilihan**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Lalai**: `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Pilihan**: Senarai float

**Lalai**: `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**Pilihan**: Satu atau lebih daripada: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Lalai**: `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**Pilihan**: Senarai float; setiap float >= 1

**Lalai**: `(1, 1.5, 2)` untuk `PEA`, dan `(1, 3, 5)` sebaliknya

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

Sejauh mana ketahanan yang perlu dibina terhadap ralat. Tahap yang lebih tinggi menghasilkan keputusan yang lebih tepat dengan mengorbankan masa pemprosesan yang lebih lama. Lihat bahagian [tahap ketahanan](/guides/estimator-noise-management#resilience) dalam topik Pengurusan hingar untuk mengetahui lebih lanjut.

**Pilihan**: `0`, `1`, `2`

**Lalai**: `1`

[Dokumentasi API `resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**Pilihan**: Integer

**Lalai**: None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

Pilihan untuk dihantar semasa mensimulasikan Backend

[Dokumentasi API `simulator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Pilihan**: Senarai nama gate asas untuk diurai

**Lalai**: Set semua gate asas yang disokong oleh [simulator Qiskit Aer](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**Pilihan**: Senarai interaksi dua qubit berarah

**Lalai**: None, yang bermakna tiada kekangan sambungan (sambungan penuh).

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**Pilihan**: [Qiskit Aer NoiseModel](/guides/build-noise-models), atau representasinya

**Lalai**: None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**Pilihan**: Integer

**Lalai**: None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

Pilihan Twirling

[Dokumentasi API `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Pilihan**: True, False

**Lalai**: False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**Pilihan**: True, False

**Lalai**: True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**Pilihan**: `auto`, Integer >= 1

**Lalai**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**Pilihan**: `auto`, Integer >= 1

**Lalai**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**Pilihan**: `active`, `active-circuit`, `active-accum`, `all`

**Lalai**: `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

Pilihan eksperimental, apabila tersedia.

  </AccordionItem>

</Accordion>
<span id="options-compatibility-table"></span>
## Keserasian ciri
Ciri-ciri runtime tertentu tidak boleh digunakan bersama dalam satu kerja. Klik tab yang sesuai untuk senarai ciri yang tidak serasi dengan ciri yang dipilih:

<Tabs>

  <TabItem value="Fractional" label="Gate pecahan">
  Tidak serasi dengan:
  - Gate twirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="Gate-folding ZNE">
    Tidak serasi dengan:
  - PEA
  - PEC

  Mungkin tidak berfungsi apabila menggunakan gate tersuai.
  </TabItem>
  <TabItem value="Twirling" label="Gate twirling">
  Tidak serasi dengan gate pecahan atau dengan regangan.

  Nota lain:
  - Twirling pengukuran hanya boleh diterapkan pada pengukuran terminal.
  - Tidak berfungsi dengan entangler bukan-Clifford.

  </TabItem>

  <TabItem value="PEA" label="PEA">
    Tidak serasi dengan:
  - Gate pecahan
  - Gate-folding ZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    Tidak serasi dengan:
  - Gate pecahan
  - Gate-folding ZNE
  - PEA
  </TabItem>

</Tabs>
## Langkah seterusnya
> **Tip:** - Semak panduan [Pengenalan kepada pilihan](/guides/runtime-options-overview).
> - Cari butiran lanjut tentang kaedah `EstimatorV2` dalam [rujukan API Estimator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2).
> - Tentukan [mod pelaksanaan](/guides/execution-modes) untuk menjalankan kerja kamu.
> - Ketahui tentang [pengurusan hingar dengan Estimator](/guides/estimator-noise-management).

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>